In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import PIL
import tensorflow as tf
import opendatasets as od 
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
import pandas as pd
import os

from sklearn.preprocessing import LabelEncoder, OneHotEncoder, MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn import tree
import random
from PIL import Image
import cv2

from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout



In [2]:
# od.download(
#     "https://www.kaggle.com/datasets/andrewmvd/ocular-disease-recognition-odir5k")

data = pd.read_excel("./Multi Class Preprocessed.xlsx")

path_to_images = "./ocular-disease-recognition-odir5k/"
data.head()

,Patient ID,Patient Age,Patient Sex,Filename,Diagnosis,N,D,G,C,A,H,M,O
0,0,69,Female,0_left.jpg,cataract,0,0,0,1,0,0,0,0
1,1,57,Male,1_left.jpg,normal fundus,1,0,0,0,0,0,0,0
2,3,66,Male,3_left.jpg,normal fundus,1,0,0,0,0,0,0,0
3,4,53,Male,4_left.jpg,macular epiretinal membrane,0,0,0,0,0,0,0,1
4,5,50,Female,5_left.jpg,moderate non proliferative retinopathy,0,1,0,0,0,0,0,0


In [3]:
# Creating the target Labels 
def createTarget(df):
    df['Labels'] = [[0,0,0,0,0,0,0] for i in range (0,len(df))] #create a column for labels with 8 0s
    target_columns = ['N', 'D', 'G', 'C', 'A', 'H', 'M', 'O']
    label = [] #create empty label array
    for i in range(0, len(df)):
        for target in target_columns:
            label.append(df.loc[i, target]) #append the value for the column for each row to the label
        
        df.at[i,'Labels'] = label #update the label column with the label array
        label = [] #reset label array
            

In [4]:
createTarget(data)

In [5]:
data = data.drop(['Patient ID', 'N', 'D', 'G', 'C', 'A', 'H', 'M', 'O'], axis=1)
data

,Patient Age,Patient Sex,Filename,Diagnosis,Labels
0,69,Female,0_left.jpg,cataract,"[0, 0, 0, 1, 0, 0, 0, 0]"
1,57,Male,1_left.jpg,normal fundus,"[1, 0, 0, 0, 0, 0, 0, 0]"
2,66,Male,3_left.jpg,normal fundus,"[1, 0, 0, 0, 0, 0, 0, 0]"
3,53,Male,4_left.jpg,macular epiretinal membrane,"[0, 0, 0, 0, 0, 0, 0, 1]"
4,50,Female,5_left.jpg,moderate non proliferative retinopathy,"[0, 1, 0, 0, 0, 0, 0, 0]"
...,...,...,...,...,...
6379,58,Male,4683_right.jpg,mild nonproliferative retinopathy,"[0, 1, 0, 0, 0, 0, 0, 0]"
6380,63,Male,4686_right.jpg,proliferative diabetic retinopathy,"[0, 1, 0, 0, 0, 0, 0, 0]"
6381,42,Male,4688_right.jpg,moderate non proliferative retinopathy,"[0, 1, 0, 0, 0, 0, 0, 0]"
6382,54,Male,4689_right.jpg,normal fundus,"[1, 0, 0, 0, 0, 0, 0, 0]"


In [6]:
def binary_to_decimal(binary_list):
    """
    Converts a list of 0s and 1s (binary) to its decimal equivalent.

    :param binary_list: List of 0s and 1s.
    :return: Decimal equivalent of the binary number.
    """
    return sum(val * (2 ** idx) for idx, val in enumerate(reversed(binary_list)))

In [7]:
# Apply the function to each row
data['target'] = data['Labels'].apply(binary_to_decimal)
data

,Patient Age,Patient Sex,Filename,Diagnosis,Labels,target
0,69,Female,0_left.jpg,cataract,"[0, 0, 0, 1, 0, 0, 0, 0]",16
1,57,Male,1_left.jpg,normal fundus,"[1, 0, 0, 0, 0, 0, 0, 0]",128
2,66,Male,3_left.jpg,normal fundus,"[1, 0, 0, 0, 0, 0, 0, 0]",128
3,53,Male,4_left.jpg,macular epiretinal membrane,"[0, 0, 0, 0, 0, 0, 0, 1]",1
4,50,Female,5_left.jpg,moderate non proliferative retinopathy,"[0, 1, 0, 0, 0, 0, 0, 0]",64
...,...,...,...,...,...,...
6379,58,Male,4683_right.jpg,mild nonproliferative retinopathy,"[0, 1, 0, 0, 0, 0, 0, 0]",64
6380,63,Male,4686_right.jpg,proliferative diabetic retinopathy,"[0, 1, 0, 0, 0, 0, 0, 0]",64
6381,42,Male,4688_right.jpg,moderate non proliferative retinopathy,"[0, 1, 0, 0, 0, 0, 0, 0]",64
6382,54,Male,4689_right.jpg,normal fundus,"[1, 0, 0, 0, 0, 0, 0, 0]",128


In [8]:
# Path to your image dataset
image_dataset_path = "./ocular-disease-recognition-odir5k/ODIR-5K/ODIR-5K/Training images"
validation_path = "./ocular-disease-recognition-odir5k/ODIR-5K/ODIR-5K/Testing images"

In [9]:
data["filepath"] = data['Filename'].apply(lambda x: os.path.join(image_dataset_path, x))

In [10]:
import cv2
import numpy as np

# Function to preprocess images
def preprocess_image(image_path):
    image = cv2.imread(image_path)
    image = cv2.resize(image, (128, 128))  # Resize to the desired input size for your CNN
    return image

# Applying the function to each image path
data['image'] = data['filepath'].apply(preprocess_image)

# Normalize age and encode gender
data['Patient Age'] = data['Patient Age'] / 100  # Assuming age is less than 100
data['Patient Sex'] = data['Patient Sex'].map({'Male': 0, 'Female': 1})  # Binary encoding

In [11]:
data

,Patient Age,Patient Sex,Filename,Diagnosis,Labels,target,filepath,image
0,0.69,1,0_left.jpg,cataract,"[0, 0, 0, 1, 0, 0, 0, 0]",16,./ocular-disease-recognition-odir5k/ODIR-5K/OD...,"[[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], ..."
1,0.57,0,1_left.jpg,normal fundus,"[1, 0, 0, 0, 0, 0, 0, 0]",128,./ocular-disease-recognition-odir5k/ODIR-5K/OD...,"[[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], ..."
2,0.66,0,3_left.jpg,normal fundus,"[1, 0, 0, 0, 0, 0, 0, 0]",128,./ocular-disease-recognition-odir5k/ODIR-5K/OD...,"[[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], ..."
3,0.53,0,4_left.jpg,macular epiretinal membrane,"[0, 0, 0, 0, 0, 0, 0, 1]",1,./ocular-disease-recognition-odir5k/ODIR-5K/OD...,"[[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], ..."
4,0.50,1,5_left.jpg,moderate non proliferative retinopathy,"[0, 1, 0, 0, 0, 0, 0, 0]",64,./ocular-disease-recognition-odir5k/ODIR-5K/OD...,"[[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], ..."
...,...,...,...,...,...,...,...,...
6379,0.58,0,4683_right.jpg,mild nonproliferative retinopathy,"[0, 1, 0, 0, 0, 0, 0, 0]",64,./ocular-disease-recognition-odir5k/ODIR-5K/OD...,"[[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], ..."
6380,0.63,0,4686_right.jpg,proliferative diabetic retinopathy,"[0, 1, 0, 0, 0, 0, 0, 0]",64,./ocular-disease-recognition-odir5k/ODIR-5K/OD...,"[[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], ..."
6381,0.42,0,4688_right.jpg,moderate non proliferative retinopathy,"[0, 1, 0, 0, 0, 0, 0, 0]",64,./ocular-disease-recognition-odir5k/ODIR-5K/OD...,"[[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], ..."
6382,0.54,0,4689_right.jpg,normal fundus,"[1, 0, 0, 0, 0, 0, 0, 0]",128,./ocular-disease-recognition-odir5k/ODIR-5K/OD...,"[[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], ..."


In [89]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import numpy as np


# Example DataFrame
# df = pd.DataFrame({'age': [25, 30, 45], 'gender': ['male', 'female', 'male']})

# Scale age
scaler = StandardScaler()
ages_scaled = scaler.fit_transform(data[['Patient Age']])

# Encode gender
encoder = OneHotEncoder()
genders_encoded = encoder.fit_transform(data[['Patient Sex']]).toarray()

# Combine age and gender
# age_gender_combined = np.hstack((ages_scaled, genders_encoded))
age_gender_combined = np.column_stack((data['Patient Age'], data['Patient Sex']))
# age_gender_combined[0]
# Now, age_gender_combined is what you will use as train_age_gender in model training
age_gender_combined.shape

(6384, 2)

In [37]:
# num_classes = 8

# import tensorflow as tf
# from tensorflow.keras.models import Model
# from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, concatenate

# # Image input path
# image_input = Input(shape=(128, 128, 3))  # Adjust the shape to match your image size
# x = Conv2D(32, (3, 3), activation='relu')(image_input)
# x = MaxPooling2D((2, 2))(x)
# x = Conv2D(64, (3, 3), activation='relu')(x)
# x = MaxPooling2D((2, 2))(x)
# x = Flatten()(x)
# image_model = Model(inputs=image_input, outputs=x)

# # Numerical/Categorical input path
# numerical_input = Input(shape=(2,))  # For gender and age
# y = Dense(16, activation='relu')(numerical_input)
# numerical_model = Model(inputs=numerical_input, outputs=y)

# # Concatenate the output of the two models
# combined = concatenate([image_model.output, numerical_model.output])

# # Fully connected layers
# z = Dense(128, activation='relu')(combined)
# z = Dropout(0.5)(z)
# z = Dense(64, activation='relu')(z)
# z = Dropout(0.5)(z)

# # Output layer
# output = Dense(num_classes, activation='softmax')(z)  # num_classes is the number of your classes

# # Final model
# model = Model(inputs=[image_model.input, numerical_model.input], outputs=output)

# # Compile the model
# model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# # Model summary
# model.summary()



Model: "model_2"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 128, 128, 3  0           []                               
                                )]                                                                
                                                                                                  
 conv2d_5 (Conv2D)              (None, 126, 126, 32  896         ['input_1[0][0]']                
                                )                                                                 
                                                                                                  
 max_pooling2d_4 (MaxPooling2D)  (None, 63, 63, 32)  0           ['conv2d_5[0][0]']               
                                                                                            

In [128]:
# from tensorflow.keras.layers import Input, Dense, Conv2D, MaxPooling2D, Flatten, Dropout, BatchNormalization, concatenate
# from tensorflow.keras.models import Model

# # Image Input Model
# image_input = Input(shape=(128, 128, 3))  # Adjust the shape to your image size
# x = Conv2D(32, (3, 3), activation='relu')(image_input)
# x = MaxPooling2D((2, 2))(x)
# x = Conv2D(64, (3, 3), activation='relu')(x)
# x = MaxPooling2D((2, 2))(x)
# x = Flatten()(x)

# # Additional Features Input Model (for Age and Gender)
# additional_input = Input(shape=(2,))  # Shape is 2 for age and gender
# y = Dense(32, activation='relu')(additional_input)
# y = Dense(16, activation='relu')(y)

# # Concatenate the outputs of the two models
# combined = concatenate([x, y])

# # Add Fully Connected Layers
# z = Dense(128, activation='relu')(combined)
# z = Dropout(0.5)(z)
# z = Dense(64, activation='relu')(z)
# z = Dropout(0.5)(z)
# output = Dense(8, activation='softmax')(z)  # num_classes is the number of classes in your dataset

# model = Model(inputs=[image_input, additional_input], outputs=output)

# # Compile the model
# model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# model.summary()



num_classes = 8
img_height = 128
img_width = 128
# model = Sequential([
#   layers.Rescaling(1./255, input_shape=(img_height, img_width, 3)),
#   layers.Conv2D(16, 3, padding='same', activation='relu'),
#   layers.MaxPooling2D(),
#   layers.Conv2D(32, 3, padding='same', activation='relu'),
#   layers.MaxPooling2D(),
#   layers.Conv2D(64, 3, padding='same', activation='relu'),
#   layers.MaxPooling2D(),
#   layers.Flatten(),
#   layers.Dense(128, activation='relu'),
#   layers.Dense(num_classes)
# ])

model = Sequential([
    # Rescaling layer (normalizing the pixel values)
    layers.Rescaling(1./255, input_shape=(img_height, img_width, 3)),

    # First Convolution Block
    layers.Conv2D(16, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Dropout(0.2),  # Dropout layer for regularization

    # Second Convolution Block
    layers.Conv2D(32, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Dropout(0.2),  # Another Dropout layer

#     # Third Convolution Block
#     layers.Conv2D(64, 3, padding='same', activation='relu'),
#     layers.MaxPooling2D(),
#     layers.Dropout(0.3),  # Increasing dropout rate

    # Fourth Convolution Block
    layers.Conv2D(128, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Dropout(0.3),  # Consistent dropout rate

    # Flattening the output for the Dense layers
    layers.Flatten(),

    # Fully Connected Layer
    layers.Dense(256, activation='relu'),  # Increased the number of neurons
    layers.Dropout(0.4),  # Higher dropout rate for the dense layer

    # Output Layer (assuming num_classes is the number of classes)
    layers.Dense(num_classes, activation='softmax')  # Softmax for multi-class classification
])








# Model summary to inspect the architecture
model.summary()

Model: "sequential_5"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 rescaling_2 (Rescaling)     (None, 128, 128, 3)       0         
                                                                 
 conv2d_24 (Conv2D)          (None, 128, 128, 16)      448       
                                                                 
 max_pooling2d_23 (MaxPoolin  (None, 64, 64, 16)       0         
 g2D)                                                            
                                                                 
 dropout_23 (Dropout)        (None, 64, 64, 16)        0         
                                                                 
 conv2d_25 (Conv2D)          (None, 64, 64, 32)        4640      
                                                                 
 max_pooling2d_24 (MaxPoolin  (None, 32, 32, 32)       0         
 g2D)                                                 

In [129]:
image_data = np.array(data["image"].tolist())

labels = data["target"]

In [130]:
len(labels)

6384

In [131]:
len(image_data)

6384

In [132]:
len(labels.unique())

8

In [133]:
# train_images, val_images, train_age_gender, val_age_gender, train_labels, val_labels = train_test_split(
#     image_data, age_gender_combined, labels, test_size=0.2, random_state=42)

train_images, val_images, train_labels, val_labels, train_age_gender, val_age_gender = train_test_split(
    image_data, labels,age_gender_combined, test_size=0.3, random_state=42)

print(len(train_images))
print(len(val_images))
print(len(train_labels))
print(len(val_labels))
print(len(train_age_gender))
print(len(val_age_gender))

4468
1916
4468
1916
4468
1916


In [134]:
# train_labels = np.array(train_labels.tolist())
# val_labels = np.array(val_labels.tolist())

#model.compile(optimizer="adam",loss="categorical_crossentropy",metrics=["accuracy"])
# Training - assuming you have prepared `train_data`, `train_labels`, `val_data`, `val_labels`
# model.fit([train_images, train_age_gender], train_labels, validation_data=([val_images, val_age_gender], val_labels), epochs=10, batch_size=32)

#model.fit(train_images, train_labels, validation_data=(val_images, val_labels), epochs=10, batch_size=32)

# Assuming train_labels and val_labels are your label arrays
# and num_classes is the number of classes (39 in this case)
from tensorflow.keras.utils import to_categorical

num_classes = 8  # Update this if the number of classes is different
label_encoder = LabelEncoder()
print(len(train_labels))
train_labels = label_encoder.fit_transform(train_labels)
train_labels_one_hot = to_categorical(train_labels, num_classes)

val_labels = label_encoder.fit_transform(val_labels)
val_labels_one_hot = to_categorical(val_labels, num_classes)

len(train_labels)
len(train_images)

4468


4468

In [135]:
print(train_age_gender.shape)  # Expected to be something like (num_samples, 2)
print(val_age_gender.shape) 

(4468, 2)
(1916, 2)


In [136]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(train_images, train_labels_one_hot, validation_data=(val_images, val_labels_one_hot), epochs=15, batch_size=16)
# model.fit([train_images, train_age_gender], train_labels, validation_data=([val_images, val_age_gender], val_labels), epochs=10, batch_size=32)


Epoch 1/15
280/280 [==============================] - 97s 343ms/step - loss: 1.5348 - accuracy: 0.4899 - val_loss: 1.5079 - val_accuracy: 0.4723
Epoch 2/15
280/280 [==============================] - 96s 341ms/step - loss: 1.4899 - accuracy: 0.4924 - val_loss: 1.4681 - val_accuracy: 0.4729
Epoch 3/15
280/280 [==============================] - 97s 348ms/step - loss: 1.4567 - accuracy: 0.4953 - val_loss: 1.4297 - val_accuracy: 0.4765
Epoch 4/15
280/280 [==============================] - 96s 344ms/step - loss: 1.4178 - accuracy: 0.4987 - val_loss: 1.4398 - val_accuracy: 0.4786
Epoch 5/15
280/280 [==============================] - 97s 348ms/step - loss: 1.3934 - accuracy: 0.5027 - val_loss: 1.4227 - val_accuracy: 0.4854
Epoch 6/15
280/280 [==============================] - 95s 338ms/step - loss: 1.3665 - accuracy: 0.5045 - val_loss: 1.4226 - val_accuracy: 0.4943
Epoch 7/15
280/280 [==============================] - 97s 346ms/step - loss: 1.3431 - accuracy: 0.5069 - val_loss: 1.3866 - val_ac

In [137]:
prediction = model.predict(val_images)

In [139]:
predicted_labels = np.argmax(prediction, axis=1)



In [141]:
true_labels_indices = np.argmax(val_labels_one_hot, axis=1)

In [142]:
conf_matrix = tf.math.confusion_matrix(true_labels_indices, predicted_labels)


In [143]:
conf_matrix

<tf.Tensor: shape=(8, 8), dtype=int32, numpy=
array([[  2,   3,   0,   0,   6,   1,   5, 239],
       [  1,  37,   0,   0,   0,   0,   1,  44],
       [  0,   1,   0,   0,   0,   0,   7,  17],
       [  0,   1,   0,   1,   1,   1,   1,  69],
       [  0,   0,   0,   0,  21,   3,   1,  69],
       [  0,   0,   0,   0,   3,   3,   0,  49],
       [  1,   0,   0,   0,   1,   1,  24, 397],
       [  1,   1,   0,   0,   7,   6,  20, 870]])>

In [144]:
import numpy as np

# Assuming conf_matrix is your confusion matrix
conf_matrix = np.array(conf_matrix)  # Ensure it's a numpy array

num_classes = conf_matrix.shape[0]
f1_scores = np.zeros(num_classes)

for i in range(num_classes):
    TP = conf_matrix[i, i]
    FP = conf_matrix[:, i].sum() - TP
    FN = conf_matrix[i, :].sum() - TP

    precision = TP / (TP + FP) if (TP + FP) != 0 else 0
    recall = TP / (TP + FN) if (TP + FN) != 0 else 0

    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) != 0 else 0
    f1_scores[i] = f1

# Average F1 score
average_f1_score = np.mean(f1_scores)

# Print the F1 score for each class and the average
print("F1 Scores by Class: ", f1_scores)
print("Average F1 Score: ", average_f1_score)


F1 Scores by Class:  [0.01532567 0.58730159 0.         0.02666667 0.31578947 0.08571429
 0.09937888 0.65438135]
Average F1 Score:  0.22306973902790353


In [145]:
num_classes

8